In [1]:
# init_check.py
# Run this FIRST to verify everything is working before main usage
# python init_check.py

import sys

print("=" * 55)
print("  Second Brain — Initialization Check")
print("=" * 55)

# ── Step 1: Check Python version ──────────────────────────
print("\n[1/6] Checking Python version…")
version = sys.version_info
if version.major < 3 or version.minor < 9:
    print(f"  ❌ Python 3.9+ required. Found: {sys.version}")
    sys.exit(1)
print(f"  ✅ Python {version.major}.{version.minor}.{version.micro}")

# ── Step 2: Check all imports ─────────────────────────────
print("\n[2/6] Checking dependencies…")
dependencies = {
    "chromadb"      : "chromadb",
    "ollama"        : "ollama",
    "PyPDF2"        : "PyPDF2",
    "beautifulsoup4": "bs4",
    "requests"      : "requests",
    "networkx"      : "networkx",
}

missing = []
for package_name, import_name in dependencies.items():
    try:
        __import__(import_name)
        print(f"  ✅ {package_name}")
    except ImportError:
        print(f"  ❌ {package_name}  →  pip install {package_name}")
        missing.append(package_name)

if missing:
    print(f"\n  Install missing packages:")
    print(f"  pip install {' '.join(missing)}")
    sys.exit(1)

# ── Step 3: Check Ollama server ───────────────────────────
print("\n[3/6] Checking Ollama server…")
import requests as req
try:
    response = req.get("http://localhost:11434/api/tags", timeout=5)
    models_data = response.json()
    available_models = [m["name"] for m in models_data.get("models", [])]
    print(f"  ✅ Ollama server running")
    print(f"     Available models: {available_models or 'none pulled yet'}")
except Exception as e:
    print(f"  ❌ Ollama server not reachable: {e}")
    print("     Start with: ollama serve")
    sys.exit(1)

# ── Step 4: Check required models ─────────────────────────
print("\n[4/6] Checking required models…")
required_models = {
    "mistral:7b"       : "LLM for reasoning & answering",
    "nomic-embed-text" : "Embedding model for vector search",
}

for model, purpose in required_models.items():
    # Check if any available model starts with the base name
    base = model.split(":")[0]
    found = any(base in m for m in available_models)
    if found:
        print(f"  ✅ {model:25} ({purpose})")
    else:
        print(f"  ⚠️  {model:25} not found — pulling now…")
        import subprocess
        result = subprocess.run(
            ["ollama", "pull", model],
            capture_output=False
        )
        if result.returncode == 0:
            print(f"  ✅ {model} pulled successfully")
        else:
            print(f"  ❌ Failed to pull {model}")
            print(f"     Run manually: ollama pull {model}")

# ── Step 5: Check storage directories ────────────────────
print("\n[5/6] Checking storage directories…")
import os

storage_paths = {
    "./brain_db"          : "ChromaDB vector store",
    "./brain_data"        : "Temporary processing files",
}

for path, purpose in storage_paths.items():
    os.makedirs(path, exist_ok=True)
    if os.path.exists(path):
        print(f"  ✅ {path:25} ({purpose})")
    else:
        print(f"  ❌ Could not create {path}")

# ── Step 6: Quick smoke test ──────────────────────────────
print("\n[6/6] Running smoke test…")
try:
    import chromadb
    client = chromadb.PersistentClient(path="./brain_db")
    col = client.get_or_create_collection("smoke_test")
    col.add(
        documents=["Smoke test document"],
        ids=["smoke_001"],
        metadatas=[{"source": "test"}],
    )
    result = col.query(
        query_texts=["test"],
        n_results=1,
    )
    assert result["documents"][0][0] == "Smoke test document"
    # Clean up
    client.delete_collection("smoke_test")
    print("  ✅ ChromaDB read/write OK")
except Exception as e:
    print(f"  ❌ ChromaDB smoke test failed: {e}")
    sys.exit(1)

try:
    import ollama
    resp = ollama.embeddings(
        model="nomic-embed-text",
        prompt="hello world"
    )
    assert len(resp["embedding"]) > 0
    print("  ✅ Ollama embeddings OK")
except Exception as e:
    print(f"  ❌ Ollama embedding test failed: {e}")
    sys.exit(1)

print("\n" + "=" * 55)
print("  ✅ All checks passed — Second Brain is ready!")
print("  Run:  python main.py --demo")
print("        python main.py")
print("=" * 55)

  Second Brain — Initialization Check

[1/6] Checking Python version…
  ✅ Python 3.10.14

[2/6] Checking dependencies…
  ✅ chromadb
  ✅ ollama
  ✅ PyPDF2
  ✅ beautifulsoup4
  ✅ requests
  ✅ networkx

[3/6] Checking Ollama server…
  ✅ Ollama server running
     Available models: ['deepseek-coder:6.7b', 'medgemma1.5:latest', 'llama3:8b', 'phi3:mini', 'mistral:7b', 'nomic-embed-text:latest', 'llama3.2:latest']

[4/6] Checking required models…
  ✅ mistral:7b                (LLM for reasoning & answering)
  ✅ nomic-embed-text          (Embedding model for vector search)

[5/6] Checking storage directories…
  ✅ ./brain_db                (ChromaDB vector store)
  ✅ ./brain_data              (Temporary processing files)

[6/6] Running smoke test…
  ✅ ChromaDB read/write OK
  ✅ Ollama embeddings OK

  ✅ All checks passed — Second Brain is ready!
  Run:  python main.py --demo
        python main.py


In [2]:
# run_sequence.py
# Shows the exact order in which components initialize
# python run_sequence.py

import logging
import time

# Configure logging BEFORE importing anything else
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("startup")


def timed(label: str):
    """Simple context manager to time each init step."""
    class Timer:
        def __enter__(self):
            self.start = time.time()
            print(f"\n{'─'*50}")
            print(f"  ▶  {label}")
            return self
        def __exit__(self, *_):
            elapsed = time.time() - self.start
            print(f"  ✅ Done in {elapsed:.2f}s")
    return Timer()


# ══════════════════════════════════════════════════════
# SEQUENCE 1 — Storage Layer (must come first)
# ══════════════════════════════════════════════════════

with timed("STEP 1 — Initialize VectorStore (ChromaDB)"):
    from storage.vector_store import VectorStore

    vector_store = VectorStore(
        collection_name="second_brain",  # logical name for this KB
        db_path="./brain_db",            # where ChromaDB writes to disk
        embed_model="nomic-embed-text",  # local embedding model via Ollama
    )
    # At this point:
    #   - ChromaDB PersistentClient is open
    #   - Collection is created or loaded from disk
    #   - Existing chunks (if any) are accessible immediately
    print(f"     Chunks in store: {vector_store.count()}")


with timed("STEP 2 — Initialize KnowledgeGraph (NetworkX)"):
    from storage.graph_store import KnowledgeGraph

    knowledge_graph = KnowledgeGraph(
        path="./brain_graph.pkl"   # pickled DiGraph, auto-loaded if exists
    )
    # At this point:
    #   - Graph is loaded from disk (or created empty)
    #   - All nodes/edges from previous sessions are available
    stats = knowledge_graph.stats()
    print(f"     Nodes: {stats['nodes']}, Edges: {stats['edges']}")


with timed("STEP 3 — Initialize SummaryStore (SQLite)"):
    from storage.summary_store import SummaryStore

    summary_store = SummaryStore(
        db_path="./brain_summaries.db"   # SQLite file
    )
    # At this point:
    #   - SQLite connection is open
    #   - Tables created if they don't exist
    #   - Document registry is available
    docs = summary_store.list_documents()
    print(f"     Documents registered: {len(docs)}")


# ══════════════════════════════════════════════════════
# SEQUENCE 2 — Ingestion Layer
# ══════════════════════════════════════════════════════

with timed("STEP 4 — Initialize Ingestion Processors"):
    from ingestion.pdf_processor  import PDFProcessor
    from ingestion.web_processor  import WebProcessor
    from ingestion.text_processor import TextProcessor

    pdf_processor  = PDFProcessor(chunk_size=500, overlap=50)
    web_processor  = WebProcessor(chunk_size=400, overlap=40, timeout=15)
    text_processor = TextProcessor(chunk_size=400, overlap=40)
    # These are stateless — no network or disk I/O at init time
    print("     PDF / Web / Text processors ready")


# ══════════════════════════════════════════════════════
# SEQUENCE 3 — Agent Layer (depends on storage)
# ══════════════════════════════════════════════════════

with timed("STEP 5 — Initialize QueryAgent (RAG)"):
    from agents.query_agent import QueryAgent

    query_agent = QueryAgent(
        vector_store=vector_store,   # injected dependency
        model="mistral:7b",
        max_history_turns=6,
    )
    # At this point:
    #   - Agent holds a reference to the vector store
    #   - Conversation history is empty (fresh per session)
    print("     QueryAgent ready — history: empty")


with timed("STEP 6 — Initialize LinkAgent (Graph builder)"):
    from agents.link_agent import LinkAgent

    link_agent = LinkAgent(
        vector_store=vector_store,
        knowledge_graph=knowledge_graph,
        model="mistral:7b",
    )
    print("     LinkAgent ready")


with timed("STEP 7 — Initialize InsightAgent"):
    from agents.insight_agent import InsightAgent

    insight_agent = InsightAgent(
        vector_store=vector_store,
        summary_store=summary_store,
        knowledge_graph=knowledge_graph,
        model="mistral:7b",
    )
    print("     InsightAgent ready")


# ══════════════════════════════════════════════════════
# SEQUENCE 4 — SecondBrain Orchestrator
# ══════════════════════════════════════════════════════

with timed("STEP 8 — Assemble SecondBrain orchestrator"):
    from brain import SecondBrain

    brain = SecondBrain(
        db_path="./brain_db",
        graph_path="./brain_graph.pkl",
        summary_db="./brain_summaries.db",
        llm_model="mistral:7b",
        embed_model="nomic-embed-text",
    )
    # SecondBrain.__init__ runs steps 1–7 internally
    # (shown separately above for clarity)


# ══════════════════════════════════════════════════════
# SEQUENCE 5 — First Use
# ══════════════════════════════════════════════════════

print("\n" + "═" * 50)
print("  System fully initialized — running first use…")
print("═" * 50)

# --- 5a: Ingest text (fastest — no file I/O) ----------
with timed("STEP 9 — First ingestion (text note)"):
    result = brain.ingest_text(
        text=(
            "Neural networks are computing systems inspired by biological "
            "neural networks. They consist of layers of interconnected nodes. "
            "Backpropagation is the algorithm used to train them by adjusting "
            "weights to minimize prediction error."
        ),
        label="neural_networks_intro",
    )
    print(f"     Chunks added: {result['chunks_added']}")
    print(f"     Doc ID:       {result['doc_id']}")

# --- 5b: First query ----------------------------------
with timed("STEP 10 — First query"):
    answer = brain.ask(
        "What is backpropagation?",
        verbose=True,
    )
    print(f"\n     Answer preview: {answer[:200]}…")

# --- 5c: Final stats ----------------------------------
with timed("STEP 11 — Final stats"):
    stats = brain.stats()
    print(f"     Documents : {stats['documents_ingested']}")
    print(f"     Chunks    : {stats['total_chunks']}")
    print(f"     Graph     : {stats['graph']['nodes']} nodes")

print("\n✅ Full initialization sequence complete!")
print("   You can now use:  python main.py")

11:46:59 [INFO] storage.vector_store — VectorStore ready — collection 'second_brain' has 2 documents
11:46:59 [INFO] storage.graph_store — KnowledgeGraph loaded — 0 nodes, 0 edges
11:46:59 [INFO] storage.summary_store — SummaryStore connected to './brain_summaries.db'
11:46:59 [INFO] brain — 🧠 Initialising Second Brain…
11:46:59 [INFO] storage.vector_store — VectorStore ready — collection 'second_brain' has 2 documents
11:46:59 [INFO] storage.graph_store — KnowledgeGraph loaded — 0 nodes, 0 edges
11:46:59 [INFO] storage.summary_store — SummaryStore connected to './brain_summaries.db'
11:46:59 [INFO] brain — ✅ Second Brain ready.
11:46:59 [INFO] httpx — HTTP Request: POST http://127.0.0.1:11434/api/embeddings "HTTP/1.1 200 OK"



──────────────────────────────────────────────────
  ▶  STEP 1 — Initialize VectorStore (ChromaDB)
     Chunks in store: 2
  ✅ Done in 0.01s

──────────────────────────────────────────────────
  ▶  STEP 2 — Initialize KnowledgeGraph (NetworkX)
     Nodes: 0, Edges: 0
  ✅ Done in 0.00s

──────────────────────────────────────────────────
  ▶  STEP 3 — Initialize SummaryStore (SQLite)
     Documents registered: 2
  ✅ Done in 0.00s

──────────────────────────────────────────────────
  ▶  STEP 4 — Initialize Ingestion Processors
     PDF / Web / Text processors ready
  ✅ Done in 0.00s

──────────────────────────────────────────────────
  ▶  STEP 5 — Initialize QueryAgent (RAG)
     QueryAgent ready — history: empty
  ✅ Done in 0.00s

──────────────────────────────────────────────────
  ▶  STEP 6 — Initialize LinkAgent (Graph builder)
     LinkAgent ready
  ✅ Done in 0.00s

──────────────────────────────────────────────────
  ▶  STEP 7 — Initialize InsightAgent
     InsightAgent ready
  ✅ D

11:46:59 [INFO] storage.vector_store — Stored 1 new chunks for doc 'neural_networks_intro_c9ceb359'


  ✅ 1 chunks stored.
     Chunks added: 1
     Doc ID:       neural_networks_intro_c9ceb359
  ✅ Done in 0.17s

──────────────────────────────────────────────────
  ▶  STEP 10 — First query


11:47:02 [INFO] httpx — HTTP Request: POST http://127.0.0.1:11434/api/embeddings "HTTP/1.1 200 OK"
11:47:02 [INFO] agents.query_agent — Querying model 'mistral:7b' with 3 context chunks…
11:47:04 [INFO] httpx — HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



💭 Sources: ['neural_networks_intro', 'learning_techniques', 'ml_overview']
   Relevance: ['0.60', '0.59', '0.55']

     Answer preview:  Backpropagation is an algorithm used to train neural networks (source: [1]). It adjusts weights within the network to minimize prediction error.…
  ✅ Done in 4.19s

──────────────────────────────────────────────────
  ▶  STEP 11 — Final stats
     Documents : 3
     Chunks    : 3
     Graph     : 0 nodes
  ✅ Done in 0.00s

✅ Full initialization sequence complete!
   You can now use:  python main.py
